# Modelo XGBoost — Fase 6, ablación de SMOTE

Compara tres formas de tratar el desbalance de clases con los mismos hiperparámetros del baseline de `02_modelo.ipynb`, evaluadas en el mismo val real:

- **A. Con SMOTE:** la versión con SMOTE que `01_feature_engineering.ipynb` guarda aparte (`X_train_smote.csv` / `y_train_smote.csv`): filas reales + sintéticas.
- **B. Sin SMOTE y sin pesos:** el train real, `X_train.csv` / `y_train.csv`.
- **C. Sin SMOTE con `scale_pos_weight`:** filas reales de train y `scale_pos_weight = benignos / ataques`, calculado sobre `y_train` real.

SMOTE cambia la calibración de las probabilidades (el modelo ve una proporción de ataques mucho más alta que la real), así que comparar solo con umbral 0.5 favorece o perjudica a la variante A por una razón que no tiene que ver con qué tan bien separa las clases. Por eso, además de las métricas a 0.5, se reportan métricas que no dependen del umbral: PR-AUC, ROC-AUC y el recall que se alcanza cuando se permite un máximo de 10 o de 66 falsos positivos (66 ≈ 0.1% de los benignos de val).

`X_test`/`y_test` no se cargan en este notebook. Test se reserva para la evaluación final.

### Regla de decisión (definida antes de ver los resultados)

Se prefiere una variante **sin SMOTE**. La regla de "supera" entre dos variantes, `X` contra `Y`, es:

1. PR-AUC de val de `X` mayor que la de `Y` por más de **0.0005**, y
2. recall de `X` estrictamente mayor que el de `Y` en **los dos** presupuestos de FP (10 y 66).

Con eso:

- Entre las dos variantes sin SMOTE, **C** reemplaza a **B** solo si C supera a B según la regla. Si no, gana la más simple (B), incluidos los empates.
- **A** (con SMOTE) gana solo si supera según la regla a **las dos** variantes sin SMOTE (B y C). Si no, gana la mejor variante sin SMOTE de la línea anterior.

El umbral 0.5, el mejor umbral y las demás métricas se reportan, pero no entran en la decisión.

In [1]:
import sys
from pathlib import Path

# Raíz del repo = primera carpeta hacia arriba que contenga src/.
# Así el notebook funciona sin importar desde dónde se abra el kernel.
RAIZ_REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ_REPO) not in sys.path:
    sys.path.append(str(RAIZ_REPO))

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from src.evaluation.metrics import Timer

DATA_DIR = RAIZ_REPO / "data" / "processed" / "xgboost"
print(f"Raíz del repo: {RAIZ_REPO}")
print(f"Datos:         {DATA_DIR}")
print(f"xgboost {xgb.__version__}")

Raíz del repo: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids
Datos:         C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\data\processed\xgboost
xgboost 3.3.0


### Carga de train y val

Solo train y val. El train real sale de `X_train.csv` / `y_train.csv` y la versión con SMOTE de `X_train_smote.csv` / `y_train_smote.csv`, ambos escritos por `01_feature_engineering.ipynb`. En la versión con SMOTE, las primeras `N_TRAIN_REAL` filas son el train real (SMOTE agrega las sintéticas al final); se verifica con asserts.

In [2]:
N_TRAIN_REAL = 310_529  # filas de train antes de SMOTE (ver 01_feature_engineering.ipynb)
N_ATAQUES_TRAIN_REAL = 8_230
N_VAL = 66_541
N_ATAQUES_VAL = 1_884

X_train_real = pd.read_csv(DATA_DIR / "X_train.csv")
y_train_real = pd.read_csv(DATA_DIR / "y_train.csv")["label_binaria"].to_numpy()
X_train_smote = pd.read_csv(DATA_DIR / "X_train_smote.csv")
y_train_smote = pd.read_csv(DATA_DIR / "y_train_smote.csv")["label_binaria"].to_numpy()
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv")["label_binaria"].to_numpy()

assert list(X_train_real.columns) == list(X_val.columns), "Columnas distintas entre train y val"
assert list(X_train_smote.columns) == list(X_val.columns), "Columnas distintas entre train con SMOTE y val"

assert len(X_train_real) == N_TRAIN_REAL and y_train_real.sum() == N_ATAQUES_TRAIN_REAL, (
    f"Se esperaban {N_TRAIN_REAL:,} filas y {N_ATAQUES_TRAIN_REAL:,} ataques en el train real, "
    f"hay {len(X_train_real):,} y {y_train_real.sum()}. Revisar que el notebook 01 guarde el train sin SMOTE."
)

X_train_sint, y_train_sint = X_train_smote.iloc[N_TRAIN_REAL:], y_train_smote[N_TRAIN_REAL:]

assert len(X_train_smote) > N_TRAIN_REAL, "X_train_smote no tiene filas sintéticas después de las reales"
assert (y_train_smote[:N_TRAIN_REAL] == y_train_real).all(), (
    "Las primeras filas de y_train_smote no coinciden con el train real: revisar SMOTE en el notebook 01."
)
assert y_train_smote[:N_TRAIN_REAL].sum() == N_ATAQUES_TRAIN_REAL, (
    f"Se esperaban {N_ATAQUES_TRAIN_REAL:,} ataques en las primeras {N_TRAIN_REAL:,} filas de train con SMOTE, "
    f"hay {y_train_smote[:N_TRAIN_REAL].sum()}. Las filas reales no son las primeras: revisar SMOTE en el notebook 01."
)
assert (y_train_sint == 1).all(), "Las filas sintéticas deberían ser todas ataque"
assert len(X_val) == N_VAL and y_val.sum() == N_ATAQUES_VAL, (
    f"Se esperaban {N_VAL:,} filas y {N_ATAQUES_VAL:,} ataques en val, hay {len(X_val):,} y {y_val.sum()}."
)

n_benignos_val = int((y_val == 0).sum())
print(f"train con SMOTE: X {X_train_smote.shape} | ataques: {y_train_smote.sum():,} ({y_train_smote.mean():.4%})")
print(f"train real:      X {X_train_real.shape} | ataques: {y_train_real.sum():,} ({y_train_real.mean():.4%})")
print(f"train sintético: {len(X_train_sint):,} filas, todas ataque")
print(f"val:            X {X_val.shape} | ataques: {y_val.sum():,} ({y_val.mean():.4%}) | benignos: {n_benignos_val:,}")

train con SMOTE: X (423218, 55) | ataques: 120,919 (28.5713%)
train real:      X (310529, 55) | ataques: 8,230 (2.6503%)
train sintético: 112,689 filas, todas ataque
val:            X (66541, 55) | ataques: 1,884 (2.8313%) | benignos: 64,657


### Dispositivo: GPU (CUDA) si está disponible

Mismo fit de prueba chiquito que en `02_modelo.ipynb`: se usa `device="cuda"` solo si el fit de prueba funciona.

In [3]:
def gpu_disponible_para_xgboost() -> bool:
    if not xgb.build_info().get("USE_CUDA", False):
        return False
    try:
        rng = np.random.default_rng(0)
        X_prueba = rng.normal(size=(200, 5))
        y_prueba = (X_prueba[:, 0] > 0).astype(int)
        xgb.XGBClassifier(n_estimators=2, tree_method="hist", device="cuda").fit(X_prueba, y_prueba)
        return True
    except xgb.core.XGBoostError:
        return False


DEVICE = "cuda" if gpu_disponible_para_xgboost() else "cpu"
print(f"Dispositivo para entrenar: {DEVICE}")

Dispositivo para entrenar: cuda


### Entrenamiento de las tres variantes

Hiperparámetros idénticos al baseline del 02 (`n_estimators=1000`, `early_stopping_rounds=50`, `eval_metric="aucpr"`, `tree_method="hist"`, `random_state=42`, `n_jobs=-1`). Lo único que cambia entre variantes son los datos de entrenamiento y, en C, `scale_pos_weight`. El early stopping mira val, igual que en el baseline.

`scale_pos_weight` de C sale de `y_train_real`: benignos / ataques de las filas reales, sin contar las sintéticas.

In [4]:
n_ataques_real = int(y_train_real.sum())
n_benignos_real = int((y_train_real == 0).sum())
SCALE_POS_WEIGHT = n_benignos_real / n_ataques_real
print(f"scale_pos_weight para C = {n_benignos_real:,} / {n_ataques_real:,} = {SCALE_POS_WEIGHT:.4f}")

variantes = {
    "A": {"nombre": "A. Con SMOTE", "X": X_train_smote, "y": y_train_smote, "extra": {}},
    "B": {"nombre": "B. Sin SMOTE, sin pesos", "X": X_train_real, "y": y_train_real, "extra": {}},
    "C": {"nombre": "C. Sin SMOTE, scale_pos_weight",
          "X": X_train_real, "y": y_train_real, "extra": {"scale_pos_weight": SCALE_POS_WEIGHT}},
}

modelos, probas_val, tiempos = {}, {}, {}
for clave, v in variantes.items():
    modelo = xgb.XGBClassifier(
        n_estimators=1000,
        early_stopping_rounds=50,
        eval_metric="aucpr",
        tree_method="hist",
        device=DEVICE,
        random_state=42,
        n_jobs=-1,
        **v["extra"],
    )
    with Timer() as t:
        modelo.fit(v["X"], v["y"], eval_set=[(X_val, y_val)], verbose=False)
    # Pasar a CPU antes de predecir: X_val vive en memoria de CPU.
    modelo.set_params(device="cpu")
    modelos[clave] = modelo
    probas_val[clave] = modelo.predict_proba(X_val)[:, 1]
    tiempos[clave] = t.elapsed
    print(f"{v['nombre']:<34} filas: {len(v['X']):>7,} | best_iteration: {modelo.best_iteration:>4} | "
          f"tiempo: {t.elapsed:.2f}s ({DEVICE})")

scale_pos_weight para C = 302,299 / 8,230 = 36.7313


A. Con SMOTE                       filas: 423,218 | best_iteration:   41 | tiempo: 0.98s (cuda)


B. Sin SMOTE, sin pesos            filas: 310,529 | best_iteration:   55 | tiempo: 0.84s (cuda)


C. Sin SMOTE, scale_pos_weight     filas: 310,529 | best_iteration:   52 | tiempo: 0.84s (cuda)


### Métricas en val

- **Umbral 0.5:** recall, precision, F1, FP y FN.
- **Independientes del umbral:** PR-AUC (average precision), ROC-AUC, y el recall máximo que se alcanza permitiendo a lo más 10 o 66 FP. Para esto se recorre la curva ROC (`roc_curve` sin descartar puntos intermedios) y se toma el mayor recall entre los umbrales con FP <= presupuesto.
- **Mejor umbral:** el que maximiza F1 entre los de 0.01 a 0.99 (pasos de 0.01) con recall >= 0.75, con su FP y FN. Si varios empatan en F1, `idxmax` se queda con el más bajo.

In [5]:
PRESUPUESTOS_FP = (10, 66)  # 66 ≈ 0.1% de los benignos de val
RECALL_MINIMO = 0.75
UMBRALES_FINOS = np.round(np.arange(0.01, 0.991, 0.01), 2)


def recall_con_presupuesto_fp(y, proba, max_fp):
    fpr, tpr, _ = roc_curve(y, proba, drop_intermediate=False)
    fp = np.round(fpr * (y == 0).sum()).astype(int)
    return float(tpr[fp <= max_fp].max())


def conteos(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return int(fp), int(fn)


def mejor_umbral(y, proba):
    filas = []
    for u in UMBRALES_FINOS:
        pred = (proba >= u).astype(int)
        fp, fn = conteos(y, pred)
        filas.append({
            "umbral": u,
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "FP": fp,
            "FN": fn,
        })
    tabla = pd.DataFrame(filas)
    candidatos = tabla[tabla["recall"] >= RECALL_MINIMO]
    return None if candidatos.empty else candidatos.loc[candidatos["f1"].idxmax()]


metricas = {}
for clave, v in variantes.items():
    proba = probas_val[clave]
    pred = (proba >= 0.5).astype(int)
    fp, fn = conteos(y_val, pred)
    mu = mejor_umbral(y_val, proba)
    metricas[clave] = {
        "best_iteration": modelos[clave].best_iteration,
        "tiempo_s": tiempos[clave],
        "pr_auc": average_precision_score(y_val, proba),
        "roc_auc": roc_auc_score(y_val, proba),
        "recall@0.5": recall_score(y_val, pred, zero_division=0),
        "precision@0.5": precision_score(y_val, pred, zero_division=0),
        "f1@0.5": f1_score(y_val, pred, zero_division=0),
        "FP@0.5": fp,
        "FN@0.5": fn,
        **{f"recall@FP<={k}": recall_con_presupuesto_fp(y_val, proba, k) for k in PRESUPUESTOS_FP},
        "mejor_umbral": np.nan if mu is None else mu["umbral"],
        "recall@mejor": np.nan if mu is None else mu["recall"],
        "f1@mejor": np.nan if mu is None else mu["f1"],
        "FP@mejor": np.nan if mu is None else mu["FP"],
        "FN@mejor": np.nan if mu is None else mu["FN"],
    }

tabla_metricas = pd.DataFrame(metricas).rename(columns={k: v["nombre"] for k, v in variantes.items()})
with pd.option_context("display.float_format", "{:.6f}".format, "display.width", 200):
    print(tabla_metricas.to_string())

                A. Con SMOTE  B. Sin SMOTE, sin pesos  C. Sin SMOTE, scale_pos_weight
best_iteration     41.000000                55.000000                       52.000000
tiempo_s            0.984375                 0.842599                        0.844347
pr_auc              0.998512                 0.994515                        0.997996
roc_auc             0.999953                 0.999917                        0.999940
recall@0.5          0.920913                 0.918259                        0.969214
precision@0.5       0.992563                 0.991972                        0.991314
f1@0.5              0.955396                 0.953693                        0.980140
FP@0.5             13.000000                14.000000                       16.000000
FN@0.5            149.000000               154.000000                       58.000000
recall@FP<=10       0.916136                 0.901274                        0.916667
recall@FP<=66       0.998938                 0.998408 

### Aplicación de la regla de decisión

Se aplica la regla definida al inicio, sin cambiarla: margen de PR-AUC de 0.0005 y recall estrictamente mayor en los dos presupuestos de FP.

In [6]:
MARGEN_PR_AUC = 0.0005
CLAVES_RECALL_FP = [f"recall@FP<={k}" for k in PRESUPUESTOS_FP]


def supera(x, y):
    """True si la variante x supera a la y según la regla de decisión."""
    mx, my = metricas[x], metricas[y]
    return (mx["pr_auc"] - my["pr_auc"] > MARGEN_PR_AUC) and all(mx[c] > my[c] for c in CLAVES_RECALL_FP)


filas_regla = []
for x, y in [("C", "B"), ("A", "B"), ("A", "C")]:
    mx, my = metricas[x], metricas[y]
    filas_regla.append({
        "comparación": f"{x} supera a {y}",
        "dif PR-AUC": mx["pr_auc"] - my["pr_auc"],
        f"PR-AUC > +{MARGEN_PR_AUC}": mx["pr_auc"] - my["pr_auc"] > MARGEN_PR_AUC,
        **{f"{c} mayor": mx[c] > my[c] for c in CLAVES_RECALL_FP},
        "supera": supera(x, y),
    })
with pd.option_context("display.float_format", "{:.6f}".format, "display.width", 200):
    print(pd.DataFrame(filas_regla).to_string(index=False))

mejor_sin_smote = "C" if supera("C", "B") else "B"
ganadora = "A" if (supera("A", "B") and supera("A", "C")) else mejor_sin_smote

print(f"\nMejor variante sin SMOTE: {variantes[mejor_sin_smote]['nombre']}")
print(f"Variante ganadora según la regla: {variantes[ganadora]['nombre']}")

 comparación  dif PR-AUC  PR-AUC > +0.0005  recall@FP<=10 mayor  recall@FP<=66 mayor  supera
C supera a B    0.003481              True                 True                 True    True
A supera a B    0.003997              True                 True                 True    True
A supera a C    0.000516              True                False                False   False

Mejor variante sin SMOTE: C. Sin SMOTE, scale_pos_weight
Variante ganadora según la regla: C. Sin SMOTE, scale_pos_weight


### Resumen

Con el dataset original, la regla eligió B (sin SMOTE, sin pesos). Con el dataset corregido (443,614 filas), que es el de la salida de abajo, eligió C (sin SMOTE, con `scale_pos_weight`): C supera a B y A no supera a C. Desde `03_tuning.ipynb` en adelante se entrena con la variante C; `02_modelo.ipynb` se conserva como baseline sin tratamiento del desbalance (variante B).

In [7]:
print("=" * 70)
print("RESUMEN — ablación de SMOTE (XGBoost, solo val)")
print("=" * 70)
print(f"Dispositivo: {DEVICE} | scale_pos_weight (C): {SCALE_POS_WEIGHT:.4f}")
for clave, v in variantes.items():
    m = metricas[clave]
    print(f"{v['nombre']:<34} PR-AUC {m['pr_auc']:.6f} | "
          f"recall@FP<=10 {m['recall@FP<=10']:.4f} | recall@FP<=66 {m['recall@FP<=66']:.4f} | "
          f"0.5 -> FP {m['FP@0.5']} FN {m['FN@0.5']}")
print(f"\nGanadora según la regla: {variantes[ganadora]['nombre']}")

RESUMEN — ablación de SMOTE (XGBoost, solo val)
Dispositivo: cuda | scale_pos_weight (C): 36.7313
A. Con SMOTE                       PR-AUC 0.998512 | recall@FP<=10 0.9161 | recall@FP<=66 0.9989 | 0.5 -> FP 13 FN 149
B. Sin SMOTE, sin pesos            PR-AUC 0.994515 | recall@FP<=10 0.9013 | recall@FP<=66 0.9984 | 0.5 -> FP 14 FN 154
C. Sin SMOTE, scale_pos_weight     PR-AUC 0.997996 | recall@FP<=10 0.9167 | recall@FP<=66 0.9989 | 0.5 -> FP 16 FN 58

Ganadora según la regla: C. Sin SMOTE, scale_pos_weight
